# V9-C — Fused-backward recompute: freeing the saved `a` tensor

V9-B attacks activation bytes via precision. V9-C attacks them via *not
storing*: the fused delta-scan backward currently saves the per-step
`a_t` tensor (B,T,256) fp32 for the backward pass. The backward can
recompute `a_{t+1}` elementwise from the already-stored `(ap, bp, kp, keep)`
instead — a handful of exp/sigmoid-equivalent ops per step, memory-bound-cheap.

Honest arithmetic (correcting the brief's ~60 MB): the backward saves `a`
alone — `b_t` was never saved (the bwd kernel rebuilds `v` from `u` inline).
At the training config that is 32x128x256x4B = **4 MB/layer**, x8 layers =
**~32 MB** freed. Target: step-time cost < 2%; falsifies (not worth it) if
recompute exceeds 5% of step time. Zero quality delta by construction —
the math is identical — verified by a 1e-6 grad-equivalence gate and a
500-step same-seed training run.


## 0. Setup (+ Triton)

In [1]:
import torch
import triton
print("torch", torch.__version__)
print("triton", triton.__version__)
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
device = "cuda" if torch.cuda.is_available() else "cpu"
assert torch.cuda.is_available(), "V9-C needs the T4 GPU"

def set_seed(s):
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

import math
def check_finite(tag, vals):
    for v in vals:
        if math.isnan(v) or math.isinf(v):
            print(f"NUMERICAL FAULT at {tag}: non-finite value {v} -- STOPPING", flush=True)
            raise SystemExit(f"numerical fault at {tag}")


torch 2.11.0+cu130
triton 3.6.0
cuda: True Tesla T4


## 1. Data (campaign corpus, vocab 104)

In [2]:
import os, urllib.request
os.makedirs("data", exist_ok=True)
URLS = {
    "tinyshakespeare.txt": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "alice.txt":           "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "frankenstein.txt":    "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "pride.txt":           "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
}
parts = []
for name, url in URLS.items():
    p = f"data/{name}"
    if not os.path.exists(p):
        urllib.request.urlretrieve(url, p)
    with open(p, encoding="utf-8", errors="ignore") as f:
        parts.append(f.read())
text = "\n".join(parts)
chars = sorted(set(text)); vocab = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.95 * len(data)); train_data, val_data = data[:n], data[n:]
nl_id = stoi.get("\n")
print(f"corpus {len(text)/1e6:.2f} MB, vocab {vocab}, newline_id={nl_id}")
assert nl_id is not None, "newline not in vocab!"

SEQ_, BS = 128, 32
def get_batch(split="train", bs=32):
    d = train_data if split == "train" else val_data
    i = torch.randint(0, len(d) - SEQ_ - 1, (bs,))
    x = torch.stack([d[j:j+SEQ_] for j in i]).to(device)
    y = torch.stack([d[j+1:j+SEQ_+1] for j in i]).to(device)
    return x, y


corpus 2.47 MB, vocab 104, newline_id=0


## 2. Models + Triton kernels (inlined) + V9-C recompute variant

In [3]:
"""
Linear-recurrent replacement for Transformer self-attention.

Experiment: take a standard decoder-only Transformer, rip out the
causal self-attention layer, and replace it with a *moving linear
equation* that compresses the past into a fixed-size running state:

    h_t = d * h_{t-1} + (1 - d) * (B x_t)        (linear recurrence)
    y_t = C h_t                                   (readout)

where d in (0,1)^S is a learned per-channel decay, B/C are learned
projections, and h has FIXED size S independent of sequence length.

This is O(T) time and O(1) memory w.r.t. context length, vs O(T^2)
for attention. The question: how much modeling power do we lose?
"""

import math
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Token mixing layers
# ----------------------------------------------------------------------------

class SelectiveSegmentedState(nn.Module):
    """V5: gated segmented state-space mixer.

    Replaces the static decay with two dynamic mechanisms:

    1. Dynamic Selective Gate (the model's "will"):
         g_t = sigmoid(W_g x_t + b_g)      in (0,1), per token & channel
       acts as a learned per-token filter: low-information tokens get
       g_t -> 0 (forgotten fast), salient tokens get g_t -> 1 (kept).

    2. Hard Memory Reset Mask (segment boundaries):
         r_t = 1 if token t is a structural boundary (e.g. newline)
         h_t = (1 - r_t) * (g_t * h_{t-1} + (1 - g_t) * u_t)
       when r_t = 1 the state is multiplied by zero: the past memory
       buffer is completely destroyed and the new segment starts fresh.
       (The residual stream still carries x_t itself forward.)

    The recurrence stays affine in h (h_t = a_t*h_{t-1} + b_t with
    a_t = (1-r_t)*g_t, b_t = (1-r_t)*(1-g_t)*u_t), so the parallel
    associative scan still applies: O(T) work, log2(T) parallel passes.
    State dim stays frozen at `state_dim` (VRAM edge preserved).
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)  # bias = per-channel prior
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        # x: (B,T,dim); reset: (B,T) binary, 1 = boundary token
        u = self.in_proj(x)                    # (B,T,S)
        g = torch.sigmoid(self.gate_proj(x))   # (B,T,S) dynamic gate
        if reset is not None:
            keep = (1.0 - reset).unsqueeze(-1)  # (B,T,1)
            a = keep * g
            b = keep * (1.0 - g) * u
        else:
            a = g
            b = (1.0 - g) * u
        h_seq = parallel_scan_ab(a, b)          # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention (the baseline)."""
    def __init__(self, dim, n_heads, dropout=0.0):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                       # (B,T,H,D)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B,H,T,D)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)


def parallel_scan_ab(A, B):
    """Parallel associative scan (Hillis-Steele) for h_t = A_t*h_{t-1} + B_t.

    A, B: (B, T, S) per-step coefficients. Returns H: (B, T, S).
    log2(T) fully-parallel passes instead of T sequential steps.
    """
    Bsz, T, S = A.shape
    A = A.contiguous()
    Bh = B.contiguous()
    idx = torch.arange(T, device=A.device).view(1, T, 1)
    o = 1
    while o < T:
        m = idx >= o
        A_sh = torch.roll(A, o, dims=1)
        B_sh = torch.roll(Bh, o, dims=1)
        # compose segment [t-o, t-1] then [t]:
        #   a_new = a_t * a_{t-o},  b_new = a_t * b_{t-o} + b_t
        A_new = A * A_sh
        B_new = A * B_sh + Bh
        A = torch.where(m, A_new, A)
        Bh = torch.where(m, B_new, Bh)
        o *= 2
    return Bh


def parallel_linear_recurrence(u, decay):
    """Parallel associative scan (Hillis-Steele) for h_t = d*h_{t-1} + c_t.

    Reformulates the sequential loop as a tree reduction:
      - each step is an affine map  h -> a*h + b, packed as pair (a, b)
      - pair composition is associative:
            (a1,b1) then (a2,b2)  =  (a2*a1, a2*b1 + b2)
      - a parallel prefix over pairs yields every h_t at once

    log2(T) fully-parallel passes instead of T sequential steps.
    u: (B, T, S) inputs already projected; decay: (S,) in (0, 1).
    Returns h: (B, T, S) — identical math to the for-loop version.
    """
    Bsz, T, S = u.shape
    c = (1.0 - decay) * u
    A = decay.expand(Bsz, T, S).contiguous()   # a-coefficients
    return parallel_scan_ab(A, c)


class LinearRunningState(nn.Module):
    """Attention replacement: fixed-size running state, linear recurrence.

    h_t = decay * h_{t-1} + (1 - decay) * (B x_t)
    y_t = C h_t

    - decay: learned per-state-channel forget factor in (0, 1)
    - B:     input  projection  dim -> state_dim
    - C:     output projection  state_dim -> dim
    State size is FIXED (does not grow with context length).
    mode="loop": native Python for-loop (sequential, slow on GPU)
    mode="scan": parallel associative scan (log2(T) parallel passes)
    """
    def __init__(self, dim, state_dim=None, dropout=0.0, mode="loop"):
        super().__init__()
        self.mode = mode
        self.state_dim = state_dim or dim
        # init decay ~ U(0.9, 0.999) -> long-ish memory at start
        init = torch.empty(self.state_dim).uniform_(math.log(0.9 / 0.1),
                                                   math.log(0.999 / 0.001))
        self.logit_decay = nn.Parameter(init)
        self.in_proj = nn.Linear(dim, self.state_dim, bias=False)
        self.out_proj = nn.Linear(self.state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, _ = x.shape
        decay = torch.sigmoid(self.logit_decay)          # (S,) in (0,1)
        u = self.in_proj(x)                              # (B,T,S)
        if self.mode == "scan":
            h_seq = parallel_linear_recurrence(u, decay)  # (B,T,S)
        else:
            keep = 1.0 - decay
            h = torch.zeros(B, self.state_dim, device=x.device, dtype=x.dtype)
            outs = []
            for t in range(T):
                h = decay * h + keep * u[:, t]
                outs.append(h)
            h_seq = torch.stack(outs, dim=1)              # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


# ----------------------------------------------------------------------------
# Blocks & models (identical except for the mixing layer)
# ----------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self, dim, n_heads, mixer, dropout=0.0, ffn_hidden=None):
        super().__init__()
        ffn_hidden = ffn_hidden or 4 * dim
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.mix = mixer
        self.mlp = nn.Sequential(
            nn.Linear(dim, ffn_hidden), nn.GELU(),
            nn.Linear(ffn_hidden, dim), nn.Dropout(dropout))

    def forward(self, x, reset=None):
        x = x + self.mix(self.ln1(x), reset)
        x = x + self.mlp(self.ln2(x))
        return x


class TinyLM(nn.Module):
    """Decoder-only LM. mixer_fn(dim, n_heads) builds the token-mixing layer."""
    def __init__(self, vocab, dim=128, n_layers=4, n_heads=4,
                 seq_len=128, mixer_fn=None, dropout=0.0, ffn_hidden=None,
                 newline_id=None):
        super().__init__()
        self.seq_len = seq_len
        self.newline_id = newline_id  # boundary token id for reset mask
        self.tok_emb = nn.Embedding(vocab, dim)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), dropout, ffn_hidden)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(
            torch.arange(T, device=idx.device))[None]
        reset = None
        if self.newline_id is not None:
            reset = (idx == self.newline_id).float()  # (B,T) boundary mask
        for blk in self.blocks:
            x = blk(x, reset)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new, temp=1.0, top_k=None):
        for _ in range(max_new):
            ctx = idx[:, -self.seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temp, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_models(vocab, dim=128, n_layers=4, n_heads=4, seq_len=128,
                 state_dim=None, dropout=0.0):
    attn = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: CausalSelfAttention(d, h, dropout),
                  dropout=dropout)
    lin = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                 mixer_fn=lambda d, h: LinearRunningState(d, state_dim, dropout),
                 dropout=dropout)
    return attn, lin


"""(inlined) V6/V7/V8-A: Triton-fused scans.

Fuses the V5 recurrence (per the V6 execution plan):
  Step A: the 256-wide hidden state lives in SRAM registers for the whole
          sequence -- loaded once per batch element, never spilled to HBM.
  Step B: sigmoid gate, hard reset mask, and the linear update
          h = keep*g*h + keep*(1-g)*u run in registers, every timestep.
  Step C: only h_t (plus a_t, saved for the backward pass) hit HBM.
          The parallel scan's O(log T) intermediate (B,T,S) tensors vanish.

Math is IDENTICAL to V5's SelectiveSegmentedState -- only the execution
path changes. Zero new parameters.
"""
import triton
import triton.language as tl

BLOCK_S = 256  # state dim; the whole state fits in SRAM


@triton.jit
def _sel_scan_fwd_kernel(u_ptr, gp_ptr, keep_ptr, h_ptr, a_ptr,
                         T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with a_t = keep_t*g_t,
    b_t = keep_t*(1-g_t)*u_t, g_t = sigmoid(gp_t)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # Step A: state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))              # Step B: gate in registers
        at = k * g
        h = at * h + k * (1.0 - g) * u_t             # Step B: update in registers
        tl.store(h_b + t * BLOCK + offs, h)          # Step C: outputs to HBM
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _sel_scan_bwd_kernel(dh_ptr, hs_ptr, ap_ptr, u_ptr, gp_ptr, keep_ptr,
                         du_ptr, dgp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    ap is (B,T+1,S) with ap[t+1] = a_{t+1} (ap[T] = 0).
    D_t = gout_t + a_{t+1}*D_{t+1}; du_t = k(1-g)D_t;
    dgp_t = k*g*(1-g)*(h_{t-1}-u_t)*D_t."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    ap_b = ap_ptr + b * bsp
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dgp_b = dgp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(ap_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))
        omg = 1.0 - g
        tl.store(du_b + t * BLOCK + offs, k * omg * D)
        tl.store(dgp_b + t * BLOCK + offs, k * g * omg * (h_prev - u_t) * D)
        dnext = D


class _SelScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, gp, keep):
        assert u.is_cuda and u.dtype == torch.float32, "V6 kernel: CUDA fp32"
        assert u.shape == gp.shape and u.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, gp, keep = u.contiguous(), gp.contiguous(), keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _sel_scan_fwd_kernel[(B,)](u, gp, keep, h, a, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, gp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, gp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h    # hs[t] = h_{t-1}
        ap = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        ap[:, :T] = a    # ap[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dgp = torch.empty_like(gp)
        _sel_scan_bwd_kernel[(B,)](dh, hs, ap, u, gp, keep, du, dgp, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        return du, dgp, None


def triton_selective_scan(u, gp, keep):
    """Fused selective scan. u, gp: (B,T,256) fp32 CUDA; keep: (B,T) fp32.
    Returns h: (B,T,256). Differentiable."""
    return _SelScanFn.apply(u, gp, keep)


class SelectiveSegmentedStateTriton(nn.Module):
    """V6 mixer: IDENTICAL math and parameters to V5's SelectiveSegmentedState.

    The only change is execution: the recurrence runs in the fused Triton
    kernel (Step A/B/C) instead of the PyTorch Hillis-Steele scan.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V6 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = self.gate_proj(x).float().contiguous()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)
        return self.dropout(self.out_proj(h_seq.to(x.dtype)))


class SelectiveSegmentedStateV7(nn.Module):
    """V7 mixer: V6's Triton-fused selective scan + two literature grafts.

    Graft #1 — output gate + state normalization (HGRN2 / Mamba-2 / GDN):
        y_t = C · (o_t ⊙ RMSNorm(h_t)),   o_t = σ(W_o x_t + b_o)
    States accumulate over segments so readout scale drifts; every strong
    post-Mamba block gates the readout. RMSNorm here is parameter-free
    (no scale vector) to keep the parameter budget exact.

    Graft #3 — layer-graded forget floor (HGRN, NeurIPS'23):
        g_t = σ(W_g x_t + b_g + floor_l)
    floor_l is a per-layer learnable scalar, initialized monotonically
    increasing bottom→top, so low layers forget fast (spelling-level) and
    top layers retain (phrase-level). Complements the newline hard reset.

    The recurrence math is UNCHANGED → the same fused Triton kernel as V6.
    With FFN 1023 the budget lands EXACTLY on the attention baseline.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V7 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)          # (B,T,S)
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


# ----------------------------------------------------------------------------
# V8-A: diagonal gated delta-rule scan
# ----------------------------------------------------------------------------

@triton.jit
def _delta_scan_fwd_kernel(u_ptr, ap_ptr, bp_ptr, kp_ptr, keep_ptr,
                           h_ptr, a_ptr, T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with
        alpha = sigmoid(ap_t), beta = sigmoid(bp_t), kk = sigmoid(kp_t),
        v = silu(u_t),
        a_t = keep_t * alpha * (1 - beta*kk^2),
        b_t = keep_t * beta * kk * v.
    Diagonal restriction of Gated DeltaNet's S_t = a_t S_{t-1}(I-b_t k k^T)
    + b_t v k^T to per-channel 1x1 systems. Strictly contractive:
    sigmoid bounds keep (1-beta*kk^2) in (0,1)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su                                 # SiLU in registers
        at = kr * alpha * (1.0 - beta * kk * kk)
        bt = kr * beta * kk * v
        h = at * h + bt
        tl.store(h_b + t * BLOCK + offs, h)
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _delta_scan_bwd_kernel(dh_ptr, hs_ptr, apn_ptr, u_ptr, ap_ptr, bp_ptr,
                           kp_ptr, keep_ptr, du_ptr, dap_ptr, dbp_ptr,
                           dkp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    apn is (B,T+1,S) with apn[t+1] = a_{t+1} (apn[T] = 0).
    Adjoint: D_t = gout_t + a_{t+1}*D_{t+1}; then per-step
      du  = D*kr*beta*kk*silu'(u)
      dap = D*h_prev*kr*alpha*(1-alpha)*(1-beta*kk^2)
      dbp = D*kr*beta*(1-beta)*(kk*v - alpha*kk^2*h_prev)
      dkp = D*kr*beta*kk*(1-kk)*(v - 2*alpha*kk*h_prev)
    (hand-derived; verified against autograd on CPU before transcription)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    apn_b = apn_ptr + b * bsp
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dap_b = dap_ptr + b * bs
    dbp_b = dbp_ptr + b * bs
    dkp_b = dkp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(apn_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su
        silu_p = su * (1.0 + u_t * (1.0 - su))
        k2 = kk * kk
        omb = 1.0 - beta * k2
        tl.store(du_b + t * BLOCK + offs,
                 D * kr * beta * kk * silu_p)
        tl.store(dap_b + t * BLOCK + offs,
                 D * h_prev * kr * alpha * (1.0 - alpha) * omb)
        tl.store(dbp_b + t * BLOCK + offs,
                 D * kr * beta * (1.0 - beta) * (kk * v - alpha * k2 * h_prev))
        tl.store(dkp_b + t * BLOCK + offs,
                 D * kr * beta * kk * (1.0 - kk) * (v - 2.0 * alpha * kk * h_prev))
        dnext = D


def _delta_scan_cpu(u, ap, bp, kp, keep):
    """CPU reference for the delta scan: differentiable pure-PyTorch loop.

    Used ONLY for local (CPU) verification. On CUDA the fused Triton
    kernel above is used instead (gate-checked against this math on Colab).
    """
    B, T, S = u.shape
    kr = keep.unsqueeze(-1)
    alpha = torch.sigmoid(ap)
    beta = torch.sigmoid(bp)
    kk = torch.sigmoid(kp)
    v = u * torch.sigmoid(u)                          # SiLU
    h = torch.zeros(B, S, dtype=u.dtype, device=u.device)
    hs = []
    for t in range(T):
        a = kr[:, t] * alpha[:, t] * (1.0 - beta[:, t] * kk[:, t] ** 2)
        bb = kr[:, t] * beta[:, t] * kk[:, t] * v[:, t]
        h = a * h + bb
        hs.append(h)
    return torch.stack(hs, dim=1)


class _DeltaScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, ap, bp, kp, keep):
        assert u.is_cuda and u.dtype == torch.float32, \
            "V8-A kernel: CUDA fp32"
        for t in (u, ap, bp, kp):
            assert t.shape == u.shape and t.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, ap, bp, kp = u.contiguous(), ap.contiguous(), bp.contiguous(), kp.contiguous()
        keep = keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _delta_scan_fwd_kernel[(B,)](u, ap, bp, kp, keep, h, a, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, ap, bp, kp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, ap, bp, kp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h      # hs[t] = h_{t-1}
        apn = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        apn[:, :T] = a     # apn[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dap = torch.empty_like(ap)
        dbp = torch.empty_like(bp)
        dkp = torch.empty_like(kp)
        _delta_scan_bwd_kernel[(B,)](dh, hs, apn, u, ap, bp, kp, keep,
                                    du, dap, dbp, dkp, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        return du, dap, dbp, dkp, None


def triton_delta_scan(u, ap, bp, kp, keep):
    """Fused diagonal delta-rule scan.

    u: raw Bx (SiLU applied in-kernel); ap/bp/kp: pre-sigmoid
    alpha/beta/key projections; keep: (B,T) reset mask.
    (B,T,256) fp32; CUDA -> Triton kernel, CPU -> differentiable
    reference loop (verification only). Returns h: (B,T,256).
    """
    if u.is_cuda:
        return _DeltaScanFn.apply(u, ap, bp, kp, keep)
    return _delta_scan_cpu(u, ap, bp, kp, keep)


class SelectiveSegmentedStateV8A(nn.Module):
    """V8-A mixer: diagonal gated delta rule on V7's chassis.

    Per channel (delta rule = erase-before-write, Gated DeltaNet Eq.10
    restricted to 1x1 systems):
        alpha_t = sigmoid(W_a x_t + b_a + floor_l)   # reuse gate_proj+floor
        beta_t  = sigmoid(W_b x_t + b_b),  b_b = 0   # NEW write_proj
        k_t     = sigmoid(W_k x_t + b_k),  b_k = +2  # NEW key_proj
        v_t     = SiLU(B x_t)                        # reuse in_proj + SiLU
        a_t = alpha_t * (1 - beta_t * k_t^2)
        b_t = beta_t * k_t * v_t
        h_t = (1 - r_t) * (a_t .* h_{t-1} + b_t)
        y_t = C . (o_t .* RMSNorm(h_t)), o_t = sigmoid(W_o x_t)

    Init puts the model near V7's convex-blend regime (k~=0.88, beta~=0.5)
    and the sigmoid bounds keep every step strictly contractive.
    With FFN 766 the budget lands within the relaxed <=3000 gate.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V8-A kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)   # B
        self.gate_proj = nn.Linear(dim, state_dim)             # W_alpha
        self.write_proj = nn.Linear(dim, state_dim)            # W_beta (NEW)
        self.key_proj = nn.Linear(dim, state_dim)              # W_k (NEW)
        self.out_gate_proj = nn.Linear(dim, state_dim)         # W_o
        self.out_proj = nn.Linear(state_dim, dim, bias=False)  # C
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.write_proj.bias.data.zero_()       # b_beta = 0  -> beta ~= 0.5
        self.key_proj.bias.data.fill_(2.0)       # b_k = +2.0  -> k ~= 0.88
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6
        self.last_state = None  # stashed h_seq for stability logging (no grad)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        ap = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        bp = self.write_proj(x).float().contiguous()
        kp = self.key_proj(x).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_delta_scan(u, ap, bp, kp, keep)      # (B,T,S)
        self.last_state = h_seq.detach()  # stability logging only; no numerics change
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


# ---- V9-C: recompute-backward variant (appended; triton_kernels.py untouched) ----
@triton.jit
def _delta_scan_bwd_recompute_kernel(dh_ptr, hs_ptr, u_ptr, ap_ptr, bp_ptr,
                           kp_ptr, keep_ptr, du_ptr, dap_ptr, dbp_ptr,
                           dkp_ptr, T, BLOCK: tl.constexpr):
    # Backward identical to _delta_scan_bwd_kernel, except a_{t+1} is
    # recomputed elementwise from (ap,bp,kp,keep) instead of loaded from a
    # saved (B,T,S) tensor. Frees 4 MB/layer at the training config.
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dap_b = dap_ptr + b * bs
    dbp_b = dbp_ptr + b * bs
    dkp_b = dkp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        if t + 1 < T:                                # recompute a_{t+1}
            ap_n = tl.load(ap_b + (t + 1) * BLOCK + offs).to(tl.float32)
            bp_n = tl.load(bp_b + (t + 1) * BLOCK + offs).to(tl.float32)
            kp_n = tl.load(kp_b + (t + 1) * BLOCK + offs).to(tl.float32)
            kr_n = tl.load(k_b + (t + 1)).to(tl.float32)
            al_n = 1.0 / (1.0 + tl.exp(-ap_n))
            be_n = 1.0 / (1.0 + tl.exp(-bp_n))
            kk_n = 1.0 / (1.0 + tl.exp(-kp_n))
            a_next = kr_n * al_n * (1.0 - be_n * kk_n * kk_n)
        else:
            a_next = tl.zeros([BLOCK], dtype=tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su
        silu_p = su * (1.0 + u_t * (1.0 - su))
        k2 = kk * kk
        omb = 1.0 - beta * k2
        tl.store(du_b + t * BLOCK + offs,
                 D * kr * beta * kk * silu_p)
        tl.store(dap_b + t * BLOCK + offs,
                 D * h_prev * kr * alpha * (1.0 - alpha) * omb)
        tl.store(dbp_b + t * BLOCK + offs,
                 D * kr * beta * (1.0 - beta) * (kk * v - alpha * k2 * h_prev))
        tl.store(dkp_b + t * BLOCK + offs,
                 D * kr * beta * kk * (1.0 - kk) * (v - 2.0 * alpha * kk * h_prev))
        dnext = D


class _DeltaScanRecomputeFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, ap, bp, kp, keep):
        assert u.is_cuda and u.dtype == torch.float32, "V9-C: CUDA fp32"
        for t in (u, ap, bp, kp):
            assert t.shape == u.shape and t.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, ap, bp, kp = u.contiguous(), ap.contiguous(), bp.contiguous(), kp.contiguous()
        keep = keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _delta_scan_fwd_kernel[(B,)](u, ap, bp, kp, keep, h, a, T,
                                     BLOCK=BLOCK_S, num_warps=4)
        # NOTE: `a` is deliberately NOT saved. The backward recomputes a_{t+1}
        # elementwise from (ap, bp, kp, keep). Frees one (B,T,S) fp32 tensor.
        del a
        ctx.save_for_backward(u, ap, bp, kp, keep, h)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, ap, bp, kp, keep, h = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h      # hs[t] = h_{t-1}
        du = torch.empty_like(u)
        dap = torch.empty_like(ap)
        dbp = torch.empty_like(bp)
        dkp = torch.empty_like(kp)
        _delta_scan_bwd_recompute_kernel[(B,)](dh, hs, u, ap, bp, kp, keep,
                                     du, dap, dbp, dkp, T,
                                     BLOCK=BLOCK_S, num_warps=4)
        return du, dap, dbp, dkp, None


def triton_delta_scan_recompute(u, ap, bp, kp, keep):
    # Fused delta scan with recompute-backward. CUDA -> new kernels above;
    # CPU -> differentiable reference (verification only; no recompute needed).
    if u.is_cuda:
        return _DeltaScanRecomputeFn.apply(u, ap, bp, kp, keep)
    return _delta_scan_cpu(u, ap, bp, kp, keep)


class SelectiveSegmentedStateV9C(SelectiveSegmentedStateV8A):
    # V9-C mixer: identical architecture AND parameters to V8-A; the scan
    # backward recomputes a_t instead of saving it.
    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        ap = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        bp = self.write_proj(x).float().contiguous()
        kp = self.key_proj(x).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_delta_scan_recompute(u, ap, bp, kp, keep)
        self.last_state = h_seq.detach()  # stability logging only
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


## 3. Numerical gate — recompute-bwd grads == standard-bwd grads (< 1e-6)

In [4]:
print("=== V9-C NUMERICAL GATE: recompute-backward vs standard-backward ===")
set_seed(0)
B, T, S = 4, 128, 256
u = torch.randn(B, T, S, device=device)
ap = torch.randn(B, T, S, device=device)
bp = torch.randn(B, T, S, device=device)
kp = torch.randn(B, T, S, device=device)
keep = (torch.rand(B, T, device=device) > 0.3).float()

h_std = triton_delta_scan(u, ap, bp, kp, keep)          # compiles std kernels
h_rec = triton_delta_scan_recompute(u, ap, bp, kp, keep)  # compiles recompute bwd
e_fwd = float((h_std - h_rec).abs().max())
print(f"fwd recompute vs standard max delta: {e_fwd:.2e} (same fwd kernel; expect 0)")

def _grads(fn):
    uu = u.clone().requires_grad_(True); aa = ap.clone().requires_grad_(True)
    bb = bp.clone().requires_grad_(True); kk_ = kp.clone().requires_grad_(True)
    fn(uu, aa, bb, kk_, keep).pow(2).sum().backward()
    return [t.grad for t in (uu, aa, bb, kk_)]

g_std = _grads(triton_delta_scan)
g_rec = _grads(triton_delta_scan_recompute)
errs = {}
for name, (a, b) in zip(("du", "dap", "dbp", "dkp"), zip(g_std, g_rec)):
    errs[name] = float((a - b).abs().max())
print("recompute-vs-standard grad max deltas:",
      {k: f"{v:.2e}" for k, v in errs.items()})

# second anchor: recompute grads vs autograd through the differentiable
# reference math (the bar the original fp32 gate used)
u4 = u.clone().requires_grad_(True); ap4 = ap.clone().requires_grad_(True)
bp4 = bp.clone().requires_grad_(True); kp4 = kp.clone().requires_grad_(True)
_delta_scan_cpu(u4, ap4, bp4, kp4, keep).pow(2).sum().backward()
errs2 = {}
for name, (a, b) in zip(("du", "dap", "dbp", "dkp"),
                      zip(g_rec, (u4.grad, ap4.grad, bp4.grad, kp4.grad))):
    errs2[name] = float((a - b).abs().max())
print("recompute-vs-reference-math grad max deltas:",
      {k: f"{v:.2e}" for k, v in errs2.items()})

assert e_fwd < 1e-7, "V9-C GATE FAILED (fwd) -- STOPPING RUN"
assert all(v < 1e-6 for v in errs.values()), \
    "V9-C GATE FAILED (recompute vs standard) -- STOPPING RUN"
assert all(v < 1e-5 for v in errs2.values()), \
    "V9-C GATE FAILED (vs reference math) -- STOPPING RUN"
print("V9-C NUMERICAL GATE PASSED -- recompute backward == standard backward")


=== V9-C NUMERICAL GATE: recompute-backward vs standard-backward ===
fwd recompute vs standard max delta: 0.00e+00 (same fwd kernel; expect 0)
recompute-vs-standard grad max deltas: {'du': '0.00e+00', 'dap': '0.00e+00', 'dbp': '0.00e+00', 'dkp': '0.00e+00'}
recompute-vs-reference-math grad max deltas: {'du': '1.43e-06', 'dap': '5.96e-07', 'dbp': '9.54e-07', 'dkp': '8.34e-07'}
V9-C NUMERICAL GATE PASSED -- recompute backward == standard backward


## 4. VRAM + step-time delta (standard vs recompute, same weights)

In [5]:
import time
print("=== VRAM + STEP-TIME: standard-bwd vs recompute-bwd ===")
DIM, LAYERS, HEADS = 256, 8, 8

def _build(mixer_cls, seed):
    set_seed(seed)
    m = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_,
        mixer_fn=lambda d, h: mixer_cls(d, state_dim=256),
        ffn_hidden=766, newline_id=nl_id).to(device)
    _fl = torch.linspace(math.log(0.3/0.7), math.log(0.9/0.1), LAYERS)
    for _i, _blk in enumerate(m.blocks):
        _blk.mix.forget_floor.data.fill_(_fl[_i])
    return m

m_std = _build(SelectiveSegmentedStateV8A, 11)
m_rec = _build(SelectiveSegmentedStateV9C, 0)
m_rec.load_state_dict(m_std.state_dict())   # identical weights
opt_s = torch.optim.AdamW(m_std.parameters(), lr=3e-4)
opt_r = torch.optim.AdamW(m_rec.parameters(), lr=3e-4)

def train_step_timed(m, opt):
    m.train(); opt.zero_grad()
    xb, yb = get_batch("train")
    torch.cuda.synchronize(); t0 = time.time()
    _, loss = m(xb, yb); loss.backward(); opt.step()
    torch.cuda.synchronize()
    return (time.time() - t0) * 1000

for _ in range(5):   # warmup (kernels compile here)
    train_step_timed(m_std, opt_s); train_step_timed(m_rec, opt_r)

def peak_of(m, opt):
    torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    train_step_timed(m, opt)
    a = torch.cuda.max_memory_allocated() / 1e6
    r = torch.cuda.max_memory_reserved() / 1e6
    torch.cuda.empty_cache()
    return a, r

a_s, r_s = peak_of(m_std, opt_s)
a_r, r_r = peak_of(m_rec, opt_r)
ms_s = sum(train_step_timed(m_std, opt_s) for _ in range(50)) / 50
ms_r = sum(train_step_timed(m_rec, opt_r) for _ in range(50)) / 50
print(f"standard : peak alloc {a_s:.0f} MB / reserved {r_s:.0f} MB | step {ms_s:.2f} ms")
print(f"recompute: peak alloc {a_r:.0f} MB / reserved {r_r:.0f} MB | step {ms_r:.2f} ms")
print(f"dVRAM alloc {a_r - a_s:+.0f} MB (expect ~-32: the saved `a` tensor, 4 MB/layer x8)")
dtime = (ms_r / ms_s - 1) * 100
print(f"step-time delta {dtime:+.1f}% (target <+2%; FALSIFIES the lever if >+5%)")
VRAM_SAVED_MB = a_s - a_r
TIME_DELTA_PCT = dtime


=== VRAM + STEP-TIME: standard-bwd vs recompute-bwd ===
standard : peak alloc 908 MB / reserved 981 MB | step 62.04 ms
recompute: peak alloc 875 MB / reserved 948 MB | step 68.76 ms
dVRAM alloc -34 MB (expect ~-32: the saved `a` tensor, 4 MB/layer x8)
step-time delta +10.8% (target <+2%; FALSIFIES the lever if >+5%)


## 5. Param-diff gate — v9c vs attention (diff <= 3000)

In [6]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

set_seed(0)
attention = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_,
    mixer_fn=lambda d, h: CausalSelfAttention(d, h), newline_id=nl_id).to(device)
v9c = _build(SelectiveSegmentedStateV9C, 0)
pa, pv = count_params(attention), count_params(v9c)
print(f"attention={pa} v9c={pv} diff={abs(pa-pv)}")
print("V9-C params identical to V8-A by construction (same __init__)")
assert abs(pa - pv) <= 3000, "PARAM-DIFF GATE FAILED -- STOPPING RUN"
print("PARAM-DIFF GATE PASSED (diff <= 3000)")
del attention
torch.cuda.empty_cache()


attention=6369792 v9c=6367736 diff=2056
V9-C params identical to V8-A by construction (same __init__)
PARAM-DIFF GATE PASSED (diff <= 3000)


## 6. Short training run — 500 steps, seed 0, lockstep v8a-std vs v9c

In [7]:
print("=== 500-STEP LOCKSTEP: v8a standard-bwd vs v9c recompute-bwd (seed 0) ===")
STEPS, EVAL = 500, 250

def state_rms_norms(m):
    norms = []
    for blk in m.blocks:
        h = getattr(getattr(blk, 'mix', None), 'last_state', None)
        if h is not None:
            norms.append(float(torch.sqrt((h.float() ** 2).mean()).item()))
    return norms

def val_loss(m, n=10):
    m.eval()
    with torch.no_grad():
        return sum(float(m(*get_batch("val"))[1]) for _ in range(n)) / n

ma = _build(SelectiveSegmentedStateV8A, 0)
mc = _build(SelectiveSegmentedStateV9C, 99)
mc.load_state_dict(ma.state_dict())   # identical init; only the backward differs
opt_a = torch.optim.AdamW(ma.parameters(), lr=3e-4)
opt_c = torch.optim.AdamW(mc.parameters(), lr=3e-4)
hist = {"v8a-std": [], "v9c-rec": []}
for step in range(1, STEPS + 1):
    xb, yb = get_batch("train")          # one batch, fed to BOTH models
    for name, m, opt in (("v8a-std", ma, opt_a), ("v9c-rec", mc, opt_c)):
        m.train(); opt.zero_grad()
        _, loss = m(xb, yb); loss.backward()
        gn = float(torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0))
        check_finite(f"lockstep step {step} {name}", [float(loss), gn])
        opt.step()
    if step % EVAL == 0 or step == 1:
        va, vc = val_loss(ma), val_loss(mc)
        hist["v8a-std"].append(va); hist["v9c-rec"].append(vc)
        sn = sum(state_rms_norms(mc)) / 8
        print(f"step {step:5d} v8a-std {va:.4f} | v9c-rec {vc:.4f} "
              f"d={vc-va:+.2e} stateRMS {sn:.3f}", flush=True)

traj_d = max(abs(a - b) for a, b in zip(hist["v8a-std"], hist["v9c-rec"]))
final_c = hist["v9c-rec"][-1]
print(f"max trajectory deviation (same seed): {traj_d:.2e} (bar 1e-3)")
print(f"v9c final val @500: {final_c:.3f} (V8-A Run 1 reference @500: 1.586)")
assert traj_d <= 1e-3, "QUALITY GATE FAILED: recompute changed the trajectory -- STOPPING"
print("QUALITY GATE PASSED -- recompute backward is trajectory-identical (zero quality delta)")
if abs(final_c - 1.586) > 0.02:
    print(f"NOTE: v9c @500 ({final_c:.3f}) differs from the cross-seed 1.586 anchor "
          f"by >0.02 -- investigate, but the binding check is the same-seed identity above")


=== 500-STEP LOCKSTEP: v8a standard-bwd vs v9c recompute-bwd (seed 0) ===


/tmp/ipykernel_502/1138044473.py:29: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  check_finite(f"lockstep step {step} {name}", [float(loss), gn])


step     1 v8a-std 4.0870 | v9c-rec 4.0864 d=-5.89e-04 stateRMS 0.108
step   250 v8a-std 1.8269 | v9c-rec 1.8266 d=-3.58e-04 stateRMS 0.138
step   500 v8a-std 1.5699 | v9c-rec 1.5854 d=+1.55e-02 stateRMS 0.156
max trajectory deviation (same seed): 1.55e-02 (bar 1e-3)
v9c final val @500: 1.585 (V8-A Run 1 reference @500: 1.586)


AssertionError: QUALITY GATE FAILED: recompute changed the trajectory -- STOPPING

## 7. Verdict

In [ ]:
print("=== V9-C VERDICT ===")
print(f"numerical gate: recompute-bwd == standard-bwd < 1e-6  (PASSED in cell 3)")
print(f"VRAM saved: {VRAM_SAVED_MB:.0f} MB alloc (expect ~32)")
print(f"step-time delta: {TIME_DELTA_PCT:+.1f}% (target <+2%, falsify >+5%)")
print(f"trajectory: v9c == v8a-std to {traj_d:.2e} over 500 lockstep steps")
ok_vram = VRAM_SAVED_MB >= 20      # expect ~32; 20 is the don't-bother floor
ok_time = TIME_DELTA_PCT <= 5.0
ok_qual = traj_d <= 1e-3
print(f"VRAM bar (>=20 MB saved): {'OK' if ok_vram else 'MISS'}")
print(f"time bar (<=+5%): {'OK' if ok_time else 'MISS -- lever falsified'}")
print(f"quality bar (traj <=1e-3): {'OK' if ok_qual else 'MISS'}")
print("V9-C VERDICT:",
      "recompute ACCEPTED -- ~32 MB freed, no measurable cost"
      if (ok_vram and ok_time and ok_qual) else
      "recompute CONDITIONAL -- see flags above")


## 8. Save everything (results file + Drive best-effort + download)

In [ ]:
lines = ["V9-C fused-backward recompute -- run record",
         f"vram_saved_alloc_MB={VRAM_SAVED_MB:.0f}",
         f"step_time_delta_pct={TIME_DELTA_PCT:+.1f}",
         f"traj_dev={traj_d:.2e}",
         f"v9c final@500={final_c:.3f}"]
for i, s in enumerate([1, 250, 500]):
    lines.append(f"step {s}: v8a-std {hist['v8a-std'][i]:.4f} v9c-rec {hist['v9c-rec'][i]:.4f}")
open("/content/v9c_results.txt", "w").write("\n".join(lines))
print(open("/content/v9c_results.txt").read())
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    import shutil, datetime
    stamp = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
    shutil.copy("/content/v9c_results.txt",
                f"/content/drive/MyDrive/v9c_results_{stamp}.txt")
    print("Drive copy saved")
except Exception as e:
    print("Drive copy skipped:", e)
try:
    from google.colab import files
    files.download("/content/v9c_results.txt")
    print("download triggered")
except Exception as e:
    print("download skipped:", e)
print("SAVE DONE -- download the .ipynb via File > Download > .ipynb")


## Reading V9-C

- **Numerical gate** (cell 3) is the whole experiment in one cell: if the
  recompute backward isn't grad-identical to < 1e-6, nothing downstream
  means anything — it hard-STOPs.
- **~32 MB, not ~60**: the brief estimated (a,b); the code only ever saved
  `a`. The header corrects the record; the measured number rules.
- **Quality** is proven by same-seed trajectory identity (bit-identical
  math), not by beating a cross-seed anchor — the 1.586 line is context.
- If step-time delta lands between +2% and +5%, the lever is *acceptable
  but unimpressive*; above +5% it is falsified per the brief.
